In [1]:
"""
Wave Field Attention V3.5 - Generation Fix
============================================

THREE critical fixes for autoregressive generation:

1. ABSOLUTE POSITION MAPPING: Token i always maps to field position
   i * stride, regardless of sequence length. No more position shifting
   during generation (the root cause of garbage output in V3.0-V3.4).

2. LEFT-ALIGNED CAUSAL KERNEL: Kernel starts at position 0 (current)
   and decays backward. No center offset. Works correctly with absolute
   positions where tokens are packed in the left part of the field.

3. NO ENERGY CONSERVATION: Removed. With absolute positions, short
   sequences leave most of the field empty. Conservation was rescaling
   signal to match near-zero empty-field energy → crushing information.
   Residual connections + LayerNorm handle normalization instead.

Physics (unchanged):
- Damped wave kernel: k(t) = exp(-α*t) * cos(ω*t + φ) for t>=0
- Different heads = different fields with different wave speeds
- Static coupling = cross-head field interactions

Complexity: O(n log n) per head via FFT convolution
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class WaveFieldAttention(nn.Module):
    
    def __init__(self, embedding_dim, num_heads, latent_dim=104, field_size=512, max_seq_len=128, device='cuda'):
        super().__init__()
        
        self.embedding_dim = embedding_dim
            # Internal dimension to change it to latent space
        self.latent_dim = latent_dim 
        
        self.num_heads = num_heads
            #Head dim is chosen based on latent_dim instead of head_dim
        self.head_dim = latent_dim // num_heads
        self.field_size = field_size
        self.max_seq_len = max_seq_len
        self.device = device

            #Bcoz we are going to embed latent dim in field instead of direct embedding dim
        assert latent_dim % num_heads == 0
            #Scale down to latent dimension for both k and v
        self.qkv_proj = nn.Linear(embedding_dim, 2 * latent_dim)
        self.out_proj = nn.Linear(embedding_dim, embedding_dim)
        
        # INNOVATION 1: Wave-parameterized kernels (per head)
        self.wave_frequency = nn.Parameter(
            torch.linspace(0.3, 4.0, num_heads)
        )
        self.wave_damping = nn.Parameter(
            torch.linspace(-3.0, 0.5, num_heads)
        )
        self.wave_phase = nn.Parameter(
            torch.linspace(0, math.pi, num_heads)
        )
        
            #Latent dim to higher dim
        self.scale_up = nn.Linear(latent_dim, embedding_dim)
        
        # INNOVATION 2: Content-dependent gating (bias=2.0, starts open)
            #Change gate proj dim to latent space just for matching with field output
        self.gate_proj = nn.Linear(embedding_dim, latent_dim)
        #self.gate_proj = nn.Linear(embedding_dim, embedding_dim)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.constant_(self.gate_proj.bias, 2.0)
        
        # INNOVATION 3: Static multi-field coupling
        self.field_coupling = nn.Parameter(
            torch.eye(num_heads) + torch.randn(num_heads, num_heads) * 0.01
        )
        
        # Fixed stride for absolute position mapping
        self.register_buffer(
            'field_stride',
            torch.tensor((field_size - 1) / max(max_seq_len - 1, 1), dtype=torch.float32)
        )
        
        self.scale = math.sqrt(self.head_dim)
    
    def _build_wave_kernels(self, device):
        """
        Build LEFT-ALIGNED causal wave kernels.
        
        V3.5: kernel[0] = current position, kernel[1] = 1 step back, etc.
        No center offset. Works with absolute positions where tokens
        are packed in the left portion of the field.
        """
        G = self.field_size
        H = self.num_heads
        
        t = torch.arange(G, dtype=torch.float32, device=device)
        
        alpha = F.softplus(self.wave_damping).unsqueeze(1)
        omega = self.wave_frequency.unsqueeze(1)
        phi = self.wave_phase.unsqueeze(1)
        
        # Left-aligned: position 0 = current, position 1 = 1 step back, ...
        kernels = torch.exp(-alpha * t.unsqueeze(0)) * torch.cos(omega * t.unsqueeze(0) + phi)
        
        kernel_sum = kernels.abs().sum(dim=1, keepdim=True).clamp(min=1e-8)
        kernels = kernels / kernel_sum
        
        return torch.fft.rfft(kernels, n=2 * G)
    
    def _wave_convolve(self, field, kernel_fft):
        """Per-head wave convolution via zero-padded FFT (linear convolution)."""
        B, H, G, D = field.shape
        pad_size = 2 * G
        
        field_t = field.permute(0, 3, 1, 2).reshape(B * D, H, G)
        field_fft = torch.fft.rfft(field_t, n=pad_size)
        convolved_fft = field_fft * kernel_fft.unsqueeze(0)
        convolved = torch.fft.irfft(convolved_fft, n=pad_size)[:, :, :G]
        
        return convolved.reshape(B, D, H, G).permute(0, 2, 3, 1)
    
    def _bilinear_scatter(self, values, field_pos_float, B, H, G, head_dim, device):
        """Deposit values onto field using bilinear interpolation."""
        N = field_pos_float.shape[0]
        
        idx_lo = field_pos_float.long().clamp(0, G - 2)
        idx_hi = idx_lo + 1
        
        frac = (field_pos_float - idx_lo.float()).clamp(0, 1)
        w_lo = (1.0 - frac).view(1, 1, N, 1)
        w_hi = frac.view(1, 1, N, 1)
        
        field = torch.zeros(B, H, G, head_dim, device=device)
        
        idx_lo_exp = idx_lo.view(1, 1, N, 1).expand(B, H, -1, head_dim)
        idx_hi_exp = idx_hi.view(1, 1, N, 1).expand(B, H, -1, head_dim)
        
        field.scatter_add_(2, idx_lo_exp, values * w_lo)
        field.scatter_add_(2, idx_hi_exp, values * w_hi)
        
        return field
    
    def _bilinear_gather(self, field, field_pos_float):
        """Read from field using bilinear interpolation."""
        B, H, G, D = field.shape
        N = field_pos_float.shape[0]
        
        idx_lo = field_pos_float.long().clamp(0, G - 2)
        idx_hi = idx_lo + 1
        
        frac = (field_pos_float - idx_lo.float()).clamp(0, 1)
        w_lo = (1.0 - frac).view(1, 1, N, 1)
        w_hi = frac.view(1, 1, N, 1)
        
        idx_lo_exp = idx_lo.view(1, 1, N, 1).expand(B, H, -1, D)
        idx_hi_exp = idx_hi.view(1, 1, N, 1).expand(B, H, -1, D)
        
        val_lo = torch.gather(field, 2, idx_lo_exp)
        val_hi = torch.gather(field, 2, idx_hi_exp)
        
        return val_lo * w_lo + val_hi * w_hi
    
    def _apply_field_coupling(self, field):
        """Static multi-field coupling."""
        B, H, G, D = field.shape
        coupling = F.softmax(self.field_coupling, dim=-1)
        field_flat = field.reshape(B, H, G * D)
        coupling_exp = coupling.unsqueeze(0).expand(B, -1, -1)
        coupled = torch.bmm(coupling_exp, field_flat)
        return coupled.reshape(B, H, G, D)
    
    def forward(self, x, mask=None):
        if x.dim() == 2:
            x = x.unsqueeze(0)
            squeeze_output = True
        else:
            squeeze_output = False
        
        B, N, D = x.shape
        G = self.field_size
        H = self.num_heads
        head_dim = self.head_dim
        
        qkv = self.qkv_proj(x)
        k, v = qkv.chunk(2, dim=-1)
        
        # q = q.view(B, N, H, head_dim).transpose(1, 2)
        k = k.view(B, N, H, head_dim).transpose(1, 2)
        v = v.view(B, N, H, head_dim).transpose(1, 2)
        
        # ABSOLUTE POSITION MAPPING (V3.5)
        # Token i ALWAYS at field position i * stride, regardless of N
        seq_pos = torch.arange(N, device=x.device, dtype=torch.float32)
        field_pos_float = (seq_pos * self.field_stride).clamp(0, G - 2)
        
        # BILINEAR SCATTER
        k_mag = k.norm(dim=-1, keepdim=True)
        deposit = v * k_mag
        field = self._bilinear_scatter(deposit, field_pos_float, B, H, G, head_dim, x.device)
        
        # WAVE CONVOLUTION
        kernel_fft = self._build_wave_kernels(x.device)
        field = self._wave_convolve(field, kernel_fft)
        
        # STATIC COUPLING
        field = self._apply_field_coupling(field)
        
        # NO ENERGY CONSERVATION (V3.5: removed — destabilizes sparse fields)
        
        # CONTENT-DEPENDENT GATING
        gate = torch.sigmoid(self.gate_proj(x))
        gate = gate.view(B, N, H, head_dim).transpose(1, 2)
        
        # BILINEAR GATHER
        gathered = self._bilinear_gather(field, field_pos_float)
            #Scale back from latent dim to embedding dim
        output = gathered * gate
        
        output = output.transpose(1, 2).reshape(B, N, self.latent_dim)
        output = self.scale_up(output)
        
        output = self.out_proj(output)
        
        if squeeze_output:
            output = output.squeeze(0)
        
        return output

In [2]:
"""
Wave Field Transformer - Physics-Based Language Model
=====================================================

Field LLM V3: A genuinely new architecture that treats language as a
physical field system, not just a sequence of tokens.

What's NEW (vs any existing architecture):
1. Wave-parameterized attention (damped oscillation kernels, not arbitrary)
2. Content-dependent gating (adaptive attention patterns)
3. Multi-field coupling (cross-head field interactions)
4. Energy conservation (information can't be created/destroyed)
5. Field interference (constructive/destructive signal combination)

What this is NOT:
- Not a Mamba clone (no state space recurrence)
- Not a Hyena clone (physics-parameterized kernels, not implicit NN kernels)
- Not a standard transformer (O(n log n), not O(n²))

Complexity: O(n log n) per layer — between O(n) and O(n²)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys
import os



class SinusoidalPositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, embedding_dim, max_cache=8192):
        super().__init__()
        self.embedding_dim = embedding_dim
        pe = self._make_pe(max_cache, embedding_dim)
        self.register_buffer('pe_cache', pe)
    
    def _make_pe(self, length, dim):
        position = torch.arange(length).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe = torch.zeros(length, dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe
    
    def forward(self, seq_len, device):
        if seq_len <= self.pe_cache.shape[0]:
            return self.pe_cache[:seq_len].to(device)
        return self._make_pe(seq_len, self.embedding_dim).to(device)


class WaveFieldTransformerLayer(nn.Module):
    """
    Single layer of the Wave Field Transformer.
    
    Structure:
    1. Wave Field Attention (physics-based, content-dependent)
    2. Feed-Forward Network (standard)
    3. Pre-norm residual connections
    """
    
    def __init__(self, embedding_dim=256, num_heads=8, ffn_dim=1024,
                 field_size=512, max_seq_len=128, dropout=0.1, device='cuda'):
        super().__init__()
        
        self.attention = WaveFieldAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            field_size=field_size,
            max_seq_len=max_seq_len,
            device=device
        )
        
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embedding_dim),
            nn.Dropout(dropout)
        )
        
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm attention with residual
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)
        
        # Pre-norm FFN with residual
        ffn_out = self.ffn(self.norm2(x))
        x = x + ffn_out
        
        return x


class FieldInterferenceModule(nn.Module):
    """
    Field Interference: Models constructive and destructive combination
    of signals from different layers.
    
    In physics, when two waves meet:
    - Same phase → constructive (amplify)
    - Opposite phase → destructive (cancel)
    
    This module lets the model learn which signals to amplify and which
    to cancel, providing a physics-based information routing mechanism.
    
    Replaces the simple GlobalContextModule with interference-based mixing.
    """
    
    def __init__(self, embedding_dim, dropout=0.1, initial_temperature=-2.0):
        super().__init__()
        
        self.embedding_dim = embedding_dim
        
        # Phase detector: determines the "phase" of each position's signal
        self.local_phase_proj = nn.Linear(embedding_dim, embedding_dim)
        self.global_phase_proj = nn.Linear(embedding_dim, embedding_dim)
        
        # V3.3: diverse temperature init — sharp vs soft for different modules
        self.interference_temperature = nn.Parameter(torch.tensor(initial_temperature))
        
        # Interference gate: controls constructive vs destructive
        self.interference_gate = nn.Linear(embedding_dim * 2, embedding_dim)
        
        # Global field summary (causal cumulative mean)
        self.compress = nn.Linear(embedding_dim, embedding_dim // 4)
        self.expand = nn.Linear(embedding_dim // 4, embedding_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embedding_dim)
    
    def forward(self, x):
        """
        Apply field interference. V3.1: sharper, more selective.
        
        x: (B, N, D) — token representations
        Returns: (B, N, D) — with interference applied
        """
        B, N, D = x.shape
        
        # Compute causal global context (O(n) — cumulative mean)
        compressed = self.compress(x)
        cumsum = torch.cumsum(compressed, dim=1)
        counts = torch.arange(1, N + 1, device=x.device, dtype=x.dtype).view(1, -1, 1)
        global_ctx = self.expand(cumsum / counts)
        global_ctx = self.dropout(global_ctx)
        
        # V3.1: Separate phase projections for local and global
        local_phase = F.normalize(self.local_phase_proj(x), dim=-1)
        global_phase = F.normalize(self.global_phase_proj(global_ctx), dim=-1)
        
        # Cosine similarity between local and global phases
        phase_alignment = (local_phase * global_phase).sum(dim=-1, keepdim=True)
        
        # V3.1: Temperature-scaled sigmoid for SHARPER interference
        # Low temperature → sharp (near binary: amplify or suppress)
        # V3.0 used sqrt(D) scaling which was too soft
        temp = F.softplus(self.interference_temperature) + 0.05
        interference_strength = torch.sigmoid(phase_alignment / temp)
        
        # Gate: combine local and global
        gate_input = torch.cat([x, global_ctx], dim=-1)
        gate = torch.sigmoid(self.interference_gate(gate_input))
        
        # Apply interference: amplify aligned, suppress misaligned
        output = x + gate * global_ctx * interference_strength
        
        return output


class WaveFieldTransformer(nn.Module):
    """
    Wave Field Transformer for Language Modeling.
    
    A physics-based language model where:
    - Tokens are mapped to a continuous field
    - Information propagates via wave dynamics (not convolution)
    - Different heads = different fields with different physics
    - Fields interact through coupling
    - Energy is conserved (anti-hallucination)
    - Interference patterns route information
    
    Drop-in replacement for CausalFieldTransformer — same interface.
    """
    
    def __init__(self,
                 vocab_size=50257,
                 embedding_dim=256,
                 num_layers=6,
                 num_heads=8,
                 ffn_dim=1024,
                 field_size=512,
                 max_seq_len=2048,
                 dropout=0.1,
                 use_checkpoint=False,
                 interference_interval=3,
                 device=None):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        self.use_checkpoint = use_checkpoint
        self.interference_interval = interference_interval
        self.device = device if device is not None else (
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_encoding = SinusoidalPositionalEncoding(
            embedding_dim, max_cache=max_seq_len
        )
        self.dropout = nn.Dropout(dropout)
        
        # Wave Field Transformer layers
        self.layers = nn.ModuleList([
            WaveFieldTransformerLayer(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                field_size=field_size,
                max_seq_len=max_seq_len,
                dropout=dropout,
                device=self.device
            )
            for _ in range(num_layers)
        ])
        
        # Field Interference modules (inserted periodically)
        # V3.3: diverse temperatures — sharp early (binary decisions),
        # softer later (nuanced refinement)
        num_interference = num_layers // interference_interval
        interference_temps = [-2.0, 0.5]
        self.interference_modules = nn.ModuleList([
            FieldInterferenceModule(
                embedding_dim=embedding_dim, dropout=dropout,
                initial_temperature=interference_temps[i % len(interference_temps)]
            )
            for i in range(num_interference)
        ])
        
        # Output
        self.norm = nn.LayerNorm(embedding_dim)
        self.output_projection = nn.Linear(embedding_dim, vocab_size, bias=False)
        
        # Tie weights
        self.output_projection.weight = self.token_embedding.weight
        
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, labels=None, mask=None):
        """
        Forward pass.
        
        input_ids: (B, N) — token indices
        labels: (B, N) — target token indices (for training)
        mask: (B, N) — attention mask (optional)
        
        Returns: logits (B, N, vocab_size) and loss (if labels provided)
        """
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        
        B, N = input_ids.shape
        
        # Embeddings + positional encoding
        x = self.token_embedding(input_ids)
        pos_enc = self.positional_encoding(N, input_ids.device)
        x = x + pos_enc.unsqueeze(0)
        x = self.dropout(x)
        
        # Wave Field layers with interference
        interference_idx = 0
        for i, layer in enumerate(self.layers):
            if self.use_checkpoint and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    layer, x, mask, use_reentrant=False
                )
            else:
                x = layer(x, mask)
            
            # Apply field interference periodically
            if ((i + 1) % self.interference_interval == 0 and
                    interference_idx < len(self.interference_modules)):
                x = self.interference_modules[interference_idx](x)
                interference_idx += 1
        
        # Output
        x = self.norm(x)
        logits = self.output_projection(x)
        
        # Compute loss if labels provided
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                labels.view(-1),
                ignore_index=-100
            )
        
        return logits, loss


if __name__ == '__main__':
    print("Testing Wave Field Transformer...")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    
    model = WaveFieldTransformer(
        vocab_size=256,
        embedding_dim=256,
        num_layers=6,
        num_heads=8,
        ffn_dim=1024,
        field_size=512,
        device=device
    ).to(device)
    
    param_count = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {param_count:,}")
    
    # Test forward
    x = torch.randint(0, 256, (2, 100), device=device)
    y = torch.randint(0, 256, (2, 100), device=device)
    
    logits, loss = model(x, labels=y)
    
    print(f"Input:  {x.shape}")
    print(f"Logits: {logits.shape}")
    print(f"Loss:   {loss.item():.3f}")
    print("Wave Field Transformer works!")

Testing Wave Field Transformer...
Device: cuda
Parameters: 4,856,290
Input:  torch.Size([2, 100])
Logits: torch.Size([2, 100, 256])
Loss:   5.596
Wave Field Transformer works!


In [3]:
import torch
import torch.nn.functional as F

def test_causality():
    print("=" * 65)
    print("  CAUSALITY TEST")
    print("  Can the model see future tokens?")
    print("=" * 65)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Build a small model
    field_size = 1024
    vocab_size = 100
    model = WaveFieldTransformer(
        vocab_size=vocab_size, embedding_dim=64, num_layers=2,
        num_heads=4, ffn_dim=128, field_size=field_size,
        max_seq_len=33, dropout=0.0, use_checkpoint=False,
        interference_interval=3, device=device,
    ).to(device)
    model.eval()

    seq_len = 10

    # Test 1: Change a FUTURE token, check if PAST logits change
    print(f"\n  Test 1: Change future token, check past logits")
    print(f"  Sequence length: {seq_len}, Field size: {field_size}")

    # Original input
    input_a = torch.randint(0, vocab_size, (1, seq_len), device=device)

    # Modified input: change ONLY the last token
    input_b = input_a.clone()
    input_b[0, -1] = (input_a[0, -1] + 50) % vocab_size

    print(f"\n  Input A: {input_a[0].tolist()}")
    print(f"  Input B: {input_b[0].tolist()} (only last token changed)")

    with torch.no_grad():
        logits_a, _ = model(input_a)
        logits_b, _ = model(input_b)

    # Check: do logits at positions 0 to seq_len-2 change?
    print(f"\n  Position-by-position logit difference:")
    print(f"  {'Position':<10} {'Max Diff':>12} {'Causal?':>10}")
    print(f"  {'-'*10} {'-'*12} {'-'*10}")

    violation_found = False
    for pos in range(seq_len):
        diff = (logits_a[0, pos] - logits_b[0, pos]).abs().max().item()
        if pos < seq_len - 1:
            causal = "YES" if diff < 1e-5 else "NO — LEAK!"
            if diff >= 1e-5:
                violation_found = True
        else:
            causal = "(changed)" if diff > 0 else "unchanged"
        print(f"  {pos:<10} {diff:>12.6f} {causal:>10}")

    if violation_found:
        print(f"\n  *** CAUSALITY VIOLATION DETECTED ***")
        print(f"  Changing the LAST token affected EARLIER positions.")
        print(f"  The model can see the future through FFT circular wraparound!")
    else:
        print(f"\n  CAUSALITY OK — past positions unaffected by future changes.")

    # Test 2: Quantify the leakage per position
    print(f"\n  Test 2: Leakage strength by position")
    max_diffs = []
    for pos in range(seq_len - 1):
        diff = (logits_a[0, pos] - logits_b[0, pos]).abs().max().item()
        max_diffs.append(diff)
        bar_len = min(int(diff * 100), 50)
        bar = "█" * bar_len + "░" * (50 - bar_len)
        print(f"  Pos {pos}: {bar} {diff:.6f}")

    # Test 3: Try with different sequence lengths
    print(f"\n  Test 3: Leakage vs sequence length")
    print(f"  {'Seq Len':<10} {'Max Leakage':>15} {'Causal?':>10}")
    for test_len in [5, 10, 20, 50, 100, 128]:
        ia = torch.randint(0, vocab_size, (1, test_len), device=device)
        ib = ia.clone()
        ib[0, -1] = (ia[0, -1] + 50) % vocab_size

        with torch.no_grad():
            la, _ = model(ia)
            lb, _ = model(ib)

        max_leak = 0
        for p in range(test_len - 1):
            diff = (la[0, p] - lb[0, p]).abs().max().item()
            max_leak = max(max_leak, diff)

        causal = "OK" if max_leak < 1e-5 else f"LEAK ({max_leak:.4f})"
        print(f"  {test_len:<10} {max_leak:>15.6f} {causal:>10}")

    # Test 4: What happens with contiguous scatter (the fix)?
    print(f"\n  Test 4: Diagnosis — Token field positions")
    for test_len in [5, 10, 50, 128]:
        seq_pos = torch.arange(test_len, dtype=torch.float32)
        field_pos = (seq_pos / max(test_len - 1, 1)) * (field_size - 1)
        field_idx = field_pos.long().clamp(0, field_size - 1)
        print(f"  N={test_len:3d}: tokens at field positions {field_idx[0].item()}, "
              f"{field_idx[1].item()}, ..., {field_idx[-2].item()}, {field_idx[-1].item()} "
              f"(spread across 0-{field_size-1})")

    print(f"\n  CONCLUSION:")
    if violation_found:
        print(f"  The scatter mapping spreads tokens across the ENTIRE field (0 to {field_size-1}).")
        print(f"  FFT convolution is CIRCULAR — position 0 wraps to position {field_size-1}.")
        print(f"  This means token 0 can see token N-1 through circular wraparound.")
        print(f"  FIX: Place tokens contiguously at positions 0 to N-1 (not spread).")
        print(f"  With N << G/2, the wraparound reaches only empty positions.")
    else:
        print(f"  No causality violation detected. Architecture is correctly causal.")

    print(f"\n{'='*65}")


test_causality()

  CAUSALITY TEST
  Can the model see future tokens?

  Test 1: Change future token, check past logits
  Sequence length: 10, Field size: 1024

  Input A: [3, 73, 44, 88, 1, 51, 87, 92, 57, 19]
  Input B: [3, 73, 44, 88, 1, 51, 87, 92, 57, 69] (only last token changed)

  Position-by-position logit difference:
  Position       Max Diff    Causal?
  ---------- ------------ ----------
  0              0.000000        YES
  1              0.000000        YES
  2              0.000000        YES
  3              0.000000        YES
  4              0.000000        YES
  5              0.000000        YES
  6              0.000000        YES
  7              0.000000        YES
  8              0.000000        YES
  9              0.059416  (changed)

  CAUSALITY OK — past positions unaffected by future changes.

  Test 2: Leakage strength by position
  Pos 0: ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.000000
  Pos 1: ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.000000
  Po

In [4]:
!pip install tokenizers

In [5]:
"""
Wave Field V3.5 + BPE Tokenizer — WikiText-2 Benchmark
========================================================
Same V3.5 physics architecture, but with Byte-Level BPE tokenizer
instead of character-level FieldTokenizerV2.

BPE benefits:
- Proper word boundaries (no more "jackbourghumanism")
- Spaces as explicit tokens
- 128 tokens covers paragraphs instead of sentences
- Standard Transformer uses same BPE → fair comparison
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os
import math
import json


# ======================================================================
# STANDARD TRANSFORMER BASELINE
# ======================================================================

class StandardTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, num_layers=6,
                 num_heads=8, ffn_dim=1024, max_seq_len=256, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_embedding = nn.Embedding(max_seq_len, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embedding_dim)
        self.output_projection = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.output_projection.weight = self.token_embedding.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _generate_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, input_ids, labels=None, mask=None):
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        B, N = input_ids.shape
        positions = torch.arange(N, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.token_embedding(input_ids) + self.positional_embedding(positions)
        x = self.dropout(x)
        causal_mask = self._generate_causal_mask(N, input_ids.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        x = self.norm(x)
        logits = self.output_projection(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), labels.view(-1), ignore_index=-100)
        return logits, loss


# ======================================================================
# BPE TOKENIZER
# ======================================================================

def train_bpe_tokenizer(train_texts, vocab_size=8000):
    """Train a Byte-Level BPE tokenizer on the training data."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        min_frequency=2,
    )

    tokenizer.train_from_iterator(train_texts, trainer=trainer)
    return tokenizer


class BPEWrapper:
    """Wrapper to give BPE tokenizer same interface as FieldTokenizerV2."""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def encode(self, text):
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    def vocab_size_actual(self):
        return self.tokenizer.get_vocab_size()


# ======================================================================
# UTILITIES
# ======================================================================

def load_wikitext2():
    from datasets import load_dataset
    print("Loading WikiText-2...")
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")
    splits = {}
    for split_name, hf_split in [('train', 'train'), ('valid', 'validation'), ('test', 'test')]:
        lines = [item['text'].strip() for item in ds[hf_split]
                 if item['text'].strip() and not item['text'].strip().startswith('=')]
        splits[split_name] = lines
    return splits


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            lr = self.base_lr * (self.step_count / self.warmup_steps)
        else:
            p = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * p))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr


def encode_lines(lines, tok, max_seq_len):
    data = []
    for line in lines:
        ids = tok.encode(line)
        if len(ids) < 2:
            continue
        if len(ids) > max_seq_len:
            for s in range(0, len(ids) - 1, max_seq_len):
                c = ids[s:s + max_seq_len + 1]
                if len(c) >= 2:
                    data.append((torch.tensor(c[:-1]), torch.tensor(c[1:])))
        else:
            data.append((torch.tensor(ids[:-1]), torch.tensor(ids[1:])))
    return data


def create_batches(data, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(len(data)).tolist()
    else:
        indices = list(range(len(data)))
    batches = []
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        bx = [data[i][0] for i in batch_idx]
        by = [data[i][1] for i in batch_idx]
        ml = max(x.size(0) for x in bx)
        px = torch.zeros(len(bx), ml, dtype=torch.long, device=device)
        py = torch.full((len(by), ml), -100, dtype=torch.long, device=device)
        for i, (x, y) in enumerate(zip(bx, by)):
            px[i, :x.size(0)] = x
            py[i, :y.size(0)] = y
        batches.append((px, py))
    return batches


@torch.no_grad()
def evaluate(model, batches, vocab_size, device, use_amp=False):
    model.eval()
    tl, tc, tt, n = 0, 0, 0, 0
    for x, y in batches:
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1), ignore_index=-100)
        tl += loss.item(); n += 1
        mask = y != -100
        tc += (logits.argmax(-1)[mask] == y[mask]).sum().item()
        tt += mask.sum().item()
    model.train()
    al = tl / max(n, 1)
    return al, math.exp(min(al, 20)), tc / max(tt, 1) * 100


@torch.no_grad()
def generate_text(model, tok, seed, device, max_tokens=60,
                  temperature=0.8, top_k=50, top_p=0.92, rep_penalty=1.2):
    model.eval()
    ids = tok.encode(seed)
    if not ids:
        return seed + " [empty]"
    gen = ids.copy()
    inp = torch.tensor(ids, device=device).unsqueeze(0)
    for _ in range(max_tokens):
        logits, _ = model(inp)
        nl = logits[0, -1, :] / temperature
        for pid in set(gen[-30:]):
            if nl[pid] > 0: nl[pid] /= rep_penalty
            else: nl[pid] *= rep_penalty
        if top_k > 0:
            tv, ti = torch.topk(nl, min(top_k, nl.size(-1)))
            f = torch.full_like(nl, float('-inf'))
            f.scatter_(0, ti, tv)
        else:
            f = nl
        if top_p < 1.0:
            sl, si = torch.sort(f, descending=True)
            cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
            rm = cp > top_p; rm[1:] = rm[:-1].clone(); rm[0] = False
            f[si[rm]] = float('-inf')
        nid = torch.multinomial(F.softmax(f, dim=-1), 1).item()
        gen.append(nid)
        inp = torch.tensor([gen], device=device)
    model.train()
    return tok.decode(gen)


def train_model(model, train_data, val_data, tok, vocab_size, device,
                model_name, num_epochs=30, batch_size=32, peak_lr=0.0003,
                use_amp=True, save_dir="checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    params = sum(p.numel() for p in model.parameters())
    print(f"\n  {model_name}: {params:,} parameters")

    optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01, eps=1e-8)
    spe = math.ceil(len(train_data) / batch_size)
    scheduler = WarmupCosineScheduler(optimizer, spe * 3, spe * num_epochs, min_lr=1e-5)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    best_vl, best_vp, best_va, best_ep = float('inf'), float('inf'), 0, 0

    t0 = time.time()
    for epoch in range(1, num_epochs + 1):
        et = time.time()
        model.train()
        batches = create_batches(train_data, batch_size, device)
        tl, nb = 0, 0
        for x, y in batches:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits, _ = model(x)
                loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1), ignore_index=-100)
            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            tl += loss.item(); nb += 1
        al = tl / max(nb, 1)
        et = time.time() - et

        if epoch % 5 == 0 or epoch == 1 or epoch == num_epochs:
            vb = create_batches(val_data, batch_size, device, shuffle=False)
            vl, vp, va = evaluate(model, vb, vocab_size, device, use_amp)
            if vl < best_vl:
                best_vl, best_vp, best_va, best_ep = vl, vp, va, epoch
                torch.save(model.state_dict(), os.path.join(save_dir, "best.pt"))
                mk = " * BEST"
            else:
                mk = ""
            print(f"  Ep {epoch:3d}/{num_epochs} | Train {al:.4f} | Val {vl:.4f} PPL {vp:.1f} Acc {va:.1f}% | {et:.1f}s{mk}")
            if epoch % 10 == 0:
                sample = generate_text(model, tok, "The president of the", device, max_tokens=30)
                print(f"    Gen: {sample}")
        else:
            print(f"  Ep {epoch:3d}/{num_epochs} | Train {al:.4f} | {et:.1f}s")

    total = time.time() - t0
    model.load_state_dict(torch.load(os.path.join(save_dir, "best.pt"), weights_only=True))

    return {
        'model_name': model_name,
        'params': params,
        'best_ppl': best_vp,
        'best_acc': best_va,
        'best_epoch': best_ep,
        'total_time': total,
    }


# ======================================================================
# MAIN
# ======================================================================

def main():
    print("=" * 65)
    print("  WAVE FIELD V3.5 + BPE TOKENIZER")
    print("  Standard Transformer vs Wave Field — fair BPE comparison")
    print("=" * 65)

    splits = load_wikitext2()

    # Train BPE tokenizer
    bpe_vocab_size = 8000
    print(f"\nTraining Byte-Level BPE tokenizer (vocab={bpe_vocab_size})...")
    raw_tokenizer = train_bpe_tokenizer(splits['train'], vocab_size=bpe_vocab_size)
    tok = BPEWrapper(raw_tokenizer)
    vocab_size = tok.vocab_size_actual()
    print(f"  BPE vocab: {vocab_size} tokens")

    # Show tokenization examples
    examples = ["The president of the united states", "Scientists discovered that"]
    for ex in examples:
        ids = tok.encode(ex)
        decoded = [tok.decode([i]) for i in ids]
        print(f"  \"{ex}\" → {len(ids)} tokens: {decoded[:10]}...")

    max_seq_len = 256
    train_data = encode_lines(splits['train'], tok, max_seq_len)
    val_data = encode_lines(splits['valid'], tok, max_seq_len)
    test_data = encode_lines(splits['test'], tok, max_seq_len)

    print(f"\n  max_seq_len: {max_seq_len}")
    print(f"  Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = device.type == 'cuda'
    print(f"  Device: {device} | AMP: {use_amp}")

    num_epochs = 30
    batch_size = 32
    peak_lr = 0.0003
    field_size = 1024
    results = []

    # # ============================================================
    # # MODEL 1: Standard Transformer
    # # ============================================================
    # print(f"\n{'='*65}")
    # print("  MODEL 1: STANDARD TRANSFORMER (O(n^2))")
    # print(f"{'='*65}")

    # std_model = StandardTransformer(
    #     vocab_size=vocab_size,
    #     embedding_dim=256,
    #     num_layers=6,
    #     num_heads=8,
    #     ffn_dim=1024,
    #     max_seq_len=max_seq_len + 1,
    #     dropout=0.1,
    # ).to(device)

    # std_result = train_model(
    #     std_model, train_data, val_data, tok, vocab_size, device,
    #     "Standard Transformer", num_epochs=num_epochs, batch_size=batch_size,
    #     peak_lr=peak_lr, use_amp=use_amp, save_dir="bpe_std_checkpoints",
    # )
    # results.append(std_result)

    # print(f"\n  --- Standard Transformer Generation ---")
    # for seed in ["The president of the", "In the year", "Scientists discovered that",
    #              "The city of New York", "He was born in"]:
    #     text = generate_text(std_model, tok, seed, device, max_tokens=40)
    #     print(f"  [{seed}] -> {text}")

    # test_batches = create_batches(test_data, batch_size, device, shuffle=False)
    # std_tl, std_tp, std_ta = evaluate(std_model, test_batches, vocab_size, device, use_amp)
    # std_result['test_ppl'] = std_tp
    # std_result['test_acc'] = std_ta
    # print(f"\n  TEST: PPL {std_tp:.1f} | Acc {std_ta:.1f}%")

    # del std_model
    # torch.cuda.empty_cache()

    # ============================================================
    # MODEL 2: Wave Field V3.5
    # ============================================================
    print(f"\n{'='*65}")
    print("  MODEL 2: WAVE FIELD V3.5 (O(n log n))")
    print(f"{'='*65}")

    wave_model = WaveFieldTransformer(
        vocab_size=vocab_size,
        embedding_dim=256,
        num_layers=6,
        num_heads=8,
        ffn_dim=1024,
        field_size=field_size,
        max_seq_len=max_seq_len + 1,
        dropout=0.1,
        use_checkpoint=True,
        interference_interval=3,
        device=device,
    ).to(device)

    wave_result = train_model(
        wave_model, train_data, val_data, tok, vocab_size, device,
        "Wave Field V3.5", num_epochs=num_epochs, batch_size=batch_size,
        peak_lr=peak_lr, use_amp=use_amp, save_dir="bpe_wave_v35_checkpoints",
    )
    results.append(wave_result)

    print(f"\n  --- Wave Field V3.5 Generation ---")
    for seed in ["The president of the", "In the year", "Scientists discovered that",
                 "The city of New York", "He was born in"]:
        text = generate_text(wave_model, tok, seed, device, max_tokens=40)
        print(f"  [{seed}] -> {text}")

    test_batches = create_batches(test_data, batch_size, device, shuffle=False)
    wave_tl, wave_tp, wave_ta = evaluate(wave_model, test_batches, vocab_size, device, use_amp)
    wave_result['test_ppl'] = wave_tp
    wave_result['test_acc'] = wave_ta
    print(f"\n  TEST: PPL {wave_tp:.1f} | Acc {wave_ta:.1f}%")

    del wave_model
    torch.cuda.empty_cache()

#     # ============================================================
#     # COMPARISON
#     # ============================================================
#     print(f"\n{'='*65}")
#     print("  BPE BENCHMARK RESULTS")
#     print(f"{'='*65}")

#     print(f"\n  {'Metric':<20} {'Std Transformer':>18} {'Wave V3.5':>18} {'Winner':>10}")
#     print(f"  {'-'*20} {'-'*18} {'-'*18} {'-'*10}")

#     s_tp, w_tp = results[0]['test_ppl'], results[1]['test_ppl']
#     s_ta, w_ta = results[0]['test_acc'], results[1]['test_acc']

#     winner_p = "Wave V3.5" if w_tp < s_tp else "Standard"
#     winner_a = "Wave V3.5" if w_ta > s_ta else "Standard"
#     print(f"  {'Test PPL':<20} {s_tp:>18.1f} {w_tp:>18.1f} {winner_p:>10}")
#     print(f"  {'Test Accuracy':<20} {s_ta:>17.1f}% {w_ta:>17.1f}% {winner_a:>10}")
#     print(f"  {'Parameters':<20} {results[0]['params']:>18,} {results[1]['params']:>18,}")
#     print(f"  {'Train Time':<20} {results[0]['total_time']/60:>17.1f}m {results[1]['total_time']/60:>17.1f}m")
#     print(f"  {'Complexity':<20} {'O(n^2)':>18} {'O(n log n)':>18}")
#     print(f"  {'Tokenizer':<20} {'BPE '+str(vocab_size):>18} {'BPE '+str(vocab_size):>18}")
#     print(f"  {'Seq Length':<20} {max_seq_len:>18} {max_seq_len:>18}")

#     if w_tp < s_tp:
#         pct = (s_tp - w_tp) / s_tp * 100
#         print(f"\n  Wave V3.5 BEATS Standard Transformer by {pct:.1f}% on test PPL!")
#     else:
#         pct = (w_tp - s_tp) / s_tp * 100
#         print(f"\n  Standard beats Wave V3.5 by {pct:.1f}% — but Wave uses O(n log n)")

#     with open("bpe_benchmark_results.json", 'w') as f:
#         json.dump(results, f, indent=2)

#     print(f"\n{'='*65}")
#     print("  BENCHMARK COMPLETE")
#     print(f"{'='*65}")


if __name__ == "__main__":
    main()

  WAVE FIELD V3.5 + BPE TOKENIZER
  Standard Transformer vs Wave Field — fair BPE comparison


Loading WikiText-2...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


Training Byte-Level BPE tokenizer (vocab=8000)...



  BPE vocab: 8000 tokens
  "The president of the united states" → 7 tokens: ['The', ' president', ' of', ' the', ' un', 'ited', ' states']...
  "Scientists discovered that" → 5 tokens: ['S', 'cient', 'ists', ' discovered', ' that']...

  max_seq_len: 256
  Train: 20,032 | Val: 2,101 | Test: 2,510
  Device: cuda | AMP: True

  MODEL 2: WAVE FIELD V3.5 (O(n log n))

  Wave Field V3.5: 6,838,754 parameters
  Ep   1/30 | Train 7.7713 | Val 7.2060 PPL 1347.4 Acc 3.8% | 82.8s * BEST
  Ep   2/30 | Train 7.2158 | 82.4s
  Ep   3/30 | Train 7.1543 | 82.5s
  Ep   4/30 | Train 6.6759 | 82.9s
  Ep   5/30 | Train 6.4401 | Val 6.3621 PPL 579.5 Acc 9.6% | 82.8s * BEST
  Ep   6/30 | Train 6.3074 | 82.8s
  Ep   7/30 | Train 6.1978 | 83.3s
  Ep   8/30 | Train 6.0616 | 83.3s
  Ep   9/30 | Train 5.8985 | 83.5s
  Ep  10/30 | Train 5.7233 | Val 5.6945 PPL 297.2 Acc 13.9% | 83.2s * BEST
    Gen: The president of the World Union and the Lave , the FDP 's pr